# 🤖 Week 7: 머신러닝 모델링 & 해석
## 토양·기후 데이터로 최적 작물 예측하기 실습

---

### 📋 학습 목표
1. 머신러닝의 기본 워크플로우(전처리 → 학습 → 평가)를 이해한다
2. 분류(Classification) 문제를 정의하고 적절한 모델을 선택할 수 있다
3. 모델의 성능을 정량적으로 평가할 수 있다
4. Feature Importance를 통해 모델의 예측 근거를 해석할 수 있다

### 🗺️ 머신러닝 워크플로우
```
데이터 로딩 → 전처리(스케일링) → Train/Test 분할 → 모델 학습 → 성능 평가 → 해석
```


---
## 1. 환경 설정 및 데이터 로딩


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# scikit-learn 라이브러리
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                            accuracy_score, ConfusionMatrixDisplay)

# 모델
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

sns.set_style('whitegrid')

# 데이터 로딩

# 데이터 다운로드 및 로딩
url = "https://raw.githubusercontent.com/agtechresearch/MLapplications-graduate/refs/heads/main/dataset/Crop_Recommendation_dataset.csv"
df = pd.read_csv(url)

In [ ]:
# 데이터 정보 확인
df.head()

---
## 2. 데이터 전처리 (Preprocessing)

### 2.1 특성(X)과 타겟(y) 분리

머신러닝에서는 <입력 데이터(X, features)>와 <예측 대상(y, target)>을 분리해야 합니다.

- **X**: 토양 영양소(N, P, K) + 기후 조건(온도, 습도, pH, 강수량) → 모델의 입력
- **y**: 작물 종류(label) → 모델이 예측할 대상


In [ ]:
# TODO: X(특성)와 y(타겟)를 분리하세요
# X: label 열을 제외한 나머지
# y: label 열

# 힌트: X = df.drop('______', axis=1)
#        y = df['______']

X = ______
y = ______

In [ ]:
# X, y 데이터 크기와 변수 정보 출력
print(f"특성(X) 크기: {X.shape}")
print(f"타겟(y) 크기: {y.shape}")
print(f"\n특성 변수: {list(X.columns)}")
print(f"타겟 클래스: {y.nunique()}개 작물")

### 2.2 학습 / 테스트 데이터 분할 (Train-Test Split)

> ⚠️ **왜 데이터를 나눌까?**
> 
> 시험 문제를 풀 때, 이미 답을 알고 있는 문제로만 평가하면 실력을 알 수 없습니다. 마찬가지로, 모델도 **한 번도 보지 못한 데이터**로 평가해야 진짜 성능을 알 수 있습니다.
> 
> - **학습 데이터 (Train)**: 모델이 패턴을 학습하는 데이터 (보통 80%)
> - **테스트 데이터 (Test)**: 모델 성능을 평가하는 데이터 (보통 20%)


In [ ]:
# TODO: 데이터를 학습용(80%)과 테스트용(20%)으로 분할하세요
# 힌트: train_test_split(X, y, test_size=0.2, random_state=42)
# random_state=42는 매번 같은 결과를 얻기 위한 시드(seed) 값입니다

X_train, X_test, y_train, y_test = ______

In [ ]:
# 학습 데이터와 테스트 데이터 크기 출력
print(f"학습 데이터: {X_train.shape[0]}개 ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"테스트 데이터: {X_test.shape[0]}개 ({X_test.shape[0]/len(X)*100:.0f}%)")

### 2.3 특성 스케일링 (Feature Scaling)

> 🔑 **왜 스케일링이 필요한가?**
> 
> 각 변수의 단위와 범위가 다릅니다:
> - N: 0 ~ 140 (kg/ha)
> - pH: 3.5 ~ 9.9 (무단위)
> - rainfall: 20 ~ 300 (mm)
> 
> KNN이나 로지스틱 회귀 같은 모델은 거리(distance)를 기반으로 하므로, 값이 큰 변수가 과도한 영향을 미칩니다. 
>
> StandardScaler는 각 변수를 평균 0, 표준편차 1로 변환합니다.
> - fit_transform (Train): 기준(평균, 표준편차)을 만들고 적용
> - transform (Test/Val): 이미 만들어진 기준을 그대로 가져와서 적용만 한다.
>> ==> 결론적으로, 테스트 데이터를 학습 과정에 개입시키지 않음으로써 모델의 "일반화 성능(Generalization)"을 정확하게 측정하기 위함


In [ ]:
# TODO: StandardScaler를 사용하여 데이터를 스케일링하세요
scaler = StandardScaler()

# 힌트: 학습 데이터로 fit_transform, 테스트 데이터로 transform만!
X_train_scaled = scaler.________(X_train)
X_test_scaled = scaler.________(X_test)

In [ ]:
# 스케일링 전후 비교
print("=== 스케일링 전 (처음 3행) ===")
print(X_train[:3].values.round(2))

print("\n=== 스케일링 후 (처음 3행) ===")
print(X_train_scaled[:3].round(2))

---
## 3. 모델 학습 (Model Training)

네 가지 분류 모델을 학습시키고 비교합니다:

| 모델 | 원리 | 장점 |
|------|------|------|
| **Decision Tree** | 조건을 나무 형태로 분기 | 해석이 쉬움 |
| **Random Forest** | 여러 Decision Tree의 앙상블 | 높은 정확도, 과적합 방지 |
| **K-Nearest Neighbors (KNN)** | 가장 가까운 k개 이웃의 다수결 | 직관적 |
| **Logistic Regression** | 확률 기반 선형 분류 | 빠르고 해석 가능 |


In [ ]:
# 모델 정의
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42)
}

# 학습 및 평가
results = {}

for name, model in models.items():
    # 스케일링이 필요한 모델과 아닌 모델 구분
    if name in ['KNN (k=5)', 'Logistic Regression']:
        # TODO: 스케일링된 데이터로 모델을 학습시키세요
        # 힌트: model.fit(X_train_scaled, y_train)
        ______
        y_pred = model.predict(X_test_scaled)
    else:
        # TODO: 원본 데이터로 모델을 학습시키세요
        ______
        y_pred = model.predict(X_test)
    
    # 정확도 계산
    acc = accuracy_score(y_test, y_pred)
    results[name] = {'accuracy': acc, 'predictions': y_pred}
    print(f"{name:25s} → 정확도: {acc:.4f} ({acc*100:.1f}%)")

### 3.1 모델 성능 비교 시각화


In [ ]:
# TODO: 모델별 정확도를 막대그래프로 비교하세요
model_names = list(results.keys())
accuracies = [results[m]['accuracy'] for m in model_names]

plt.figure(figsize=(10, 5))

# 힌트: plt.barh(model_names, accuracies)
# 또는 sns.barplot(x=accuracies, y=model_names)
colors = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12']
plt.barh(model_names, accuracies, color=colors, edgecolor='black', height=0.5)

# 각 막대에 정확도 수치 표시
for i, acc in enumerate(accuracies):
    plt.text(acc + 0.002, i, f'{acc:.3f}', va='center', fontsize=12)

plt.xlabel('Accuracy')
plt.title('Model Performance Comparison', fontsize=14)
plt.xlim(0.8, 1.02)
plt.tight_layout()
plt.show()

---
## 4. 모델 평가 심화

### 4.1 혼동 행렬 (Confusion Matrix)

가장 성능이 좋은 모델의 혼동 행렬을 분석합니다. 혼동 행렬은 **어떤 작물이 어떤 작물로 잘못 예측되는지**를 보여줍니다.


In [ ]:
# 가장 정확도가 높은 모델 찾기
best_model_name = max(results, key=lambda x: results[x]['accuracy'])
best_pred = results[best_model_name]['predictions']

print(f"최고 성능 모델: {best_model_name} (정확도: {results[best_model_name]['accuracy']:.4f})")

# TODO: confusion_matrix를 계산하세요
# 힌트: cm = confusion_matrix(y_test, best_pred)
cm = ______

# 혼동 행렬 시각화
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGn',
            xticklabels=sorted(y.unique()),
            yticklabels=sorted(y.unique()), ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title(f'Confusion Matrix - {best_model_name}', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### 4.2 분류 리포트 (Classification Report)

**Precision, Recall, F1-Score**를 작물별로 확인합니다.

| 지표 | 의미 | 농업 맥락 |
|------|------|-----------|
| **Precision** | "이 작물이라 예측한 것 중 실제로 맞는 비율" | 잘못된 추천 최소화 |
| **Recall** | "실제 이 작물인 것 중 정확히 맞춘 비율" | 적합한 작물 놓치지 않기 |
| **F1-Score** | Precision과 Recall의 조화평균 | 종합 성능 |


In [ ]:
print(classification_report(y_test, best_pred))

---
## 5. 특성 중요도 (Feature Importance) - 모델 해석

> 🌱 **핵심 질문**: "작물 추천에 가장 중요한 환경 요인은 무엇인가?"

Random Forest 모델은 각 특성이 예측에 얼마나 기여했는지 알려줍니다. 이것은 "이 모델이 왜 이런 결정을 내렸는지"를 이해하는 중요한 도구입니다.


In [ ]:
# Random Forest 모델의 특성 중요도 추출
rf_model = models['Random Forest']

# TODO: feature_importances_ 속성으로 특성 중요도를 가져오세요
# 힌트: rf_model.feature_importances_
importances = ______
feature_names = X.columns

# 중요도순 정렬
feat_imp = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_imp = feat_imp.sort_values('Importance', ascending=True)

# 시각화
plt.figure(figsize=(10, 6))
plt.barh(feat_imp['Feature'], feat_imp['Importance'], color='forestgreen', edgecolor='black')
plt.xlabel('Feature Importance')
plt.title('Random Forest - Feature Importance for Crop Recommendation', fontsize=14)

# 각 막대에 수치 표시
for i, (imp, feat) in enumerate(zip(feat_imp['Importance'], feat_imp['Feature'])):
    plt.text(imp + 0.005, i, f'{imp:.3f}', va='center', fontsize=11)

plt.tight_layout()
plt.show()

#### 💡 해석 포인트

특성 중요도 결과를 보고 다음을 생각해보세요:
1. 가장 중요한 특성은 무엇인가요? 식물 재배의 입장에서 타당한가?
2. N, P, K 중 어떤 영양소가 가장 큰 영향을 미치나?
3. 기후 요인(온도, 습도, 강수량)과 토양 요인(N, P, K, pH) 중 어느 쪽이 더 중요한가?


#### ✍️ 여러분의 해석을 적어보세요
*(이 셀을 더블클릭하여 답변을 작성하세요)*

1. 가장 중요한 특성: 
2. 영양소 비교: 
3. 기후 vs 토양: 


---
## 6. 새로운 데이터로 예측하기 🌱

학습된 모델을 사용하여 **새로운 토양·기후 조건**에 적합한 작물을 예측해봅시다.


In [ ]:
# 새로운 농경지 데이터 (가상)
new_data = pd.DataFrame({
    'N': [90],       # 질소: 높음
    'P': [42],       # 인: 중간
    'K': [43],       # 칼륨: 중간
    'temperature': [21.0],  # 온도: 온대
    'humidity': [82.0],     # 습도: 높음
    'ph': [6.5],            # pH: 약산성
    'rainfall': [200.0]     # 강수량: 다습
})

# TODO: Random Forest 모델로 이 농경지에 적합한 작물을 예측하세요
# 힌트: rf_model.predict(new_data)
prediction = ______

print("="*50)
print(f"🌾 추천 작물: {prediction[0]}")
print("="*50)
print(f"\n입력 조건:")
for col in new_data.columns:
    print(f"  {col:15s}: {new_data[col].values[0]}")

### 6.1 🧪 도전: 다양한 환경 조건으로 실험해보기

아래 코드를 수정하여 알고 있는 실제 환경조건을 입력해보세요!


In [ ]:
# 여러분의 환경 조건을 입력해보세요!
# 예: 여러분 연구실의 온실 환경, 고향의 농경지 조건 등

my_data = pd.DataFrame({
    'N': [___],           # 0 ~ 140
    'P': [___],           # 5 ~ 145
    'K': [___],           # 5 ~ 205
    'temperature': [___], # 8 ~ 44 °C
    'humidity': [___],    # 14 ~ 100 %
    'ph': [___],          # 3.5 ~ 9.9
    'rainfall': [___]     # 20 ~ 300 mm
})

my_prediction = rf_model.predict(my_data)
print(f"🌾 추천 작물: {my_prediction[0]}")

# 추가: 상위 3개 추천 확률 (Random Forest가 제공하는 확률 기반)
probabilities = rf_model.predict_proba(my_data)[0]
crop_labels = rf_model.classes_
top3_idx = probabilities.argsort()[-3:][::-1]

print(f"\n📊 상위 3개 추천:")
for rank, idx in enumerate(top3_idx, 1):
    print(f"  {rank}위: {crop_labels[idx]:15s} (확률: {probabilities[idx]*100:.1f}%)")

---
## 7. 🌳 의사결정나무 시각화

Decision Tree는 모델의 **의사결정 과정을 눈으로 볼 수 있는** 유일한 모델입니다.


In [ ]:
from sklearn.tree import plot_tree

# 깊이를 3으로 제한한 간단한 Decision Tree
dt_simple = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_simple.fit(X_train, y_train)

plt.figure(figsize=(20, 10))
plot_tree(dt_simple, feature_names=list(X.columns), class_names=sorted(y.unique()),
          filled=True, rounded=True, fontsize=9, proportion=True)
plt.title('Decision Tree (max_depth=3) - Crop Recommendation', fontsize=16)
plt.tight_layout()
plt.show()

print(f"\n간소화된 Decision Tree 정확도: {dt_simple.score(X_test, y_test):.4f}")

---
## 8. 전체 과정 정리 & 토론

### 📝 실습을 통해 학습한 내용

| 실습 | 핵심 내용 | 핵심 함수/도구 |
|------|-----------|----------------|
| ** 1 ** | 데이터 탐색(EDA) | `head()`, `describe()`, `groupby()`, `corr()` |
| ** 2 ** | 데이터 시각화 | `histplot`, `boxplot`, `scatterplot`, `heatmap`, `pairplot` |
| ** 3 ** | ML 모델링 & 해석 | `train_test_split`, `fit/predict`, `confusion_matrix`, `feature_importances_` |

### 🎯 머신러닝 전체 워크플로우 요약
```
1. 데이터 수집      → CSV, 센서, DB 등
2. 탐색적 분석(EDA) → 결측치, 분포, 상관관계 파악
3. 시각화           → 패턴과 이상치 발견
4. 전처리           → 스케일링, 인코딩, 결측치 처리
5. 모델 학습        → 적절한 알고리즘 선택
6. 성능 평가        → 정확도, F1, 혼동행렬
7. 해석             → Feature Importance, 의사결정나무
8. 배포/활용        → 새로운 데이터에 적용
```

### 💬 토론 주제
1. **이 모델을 실제 농업에 적용한다면** 어떤 추가 데이터가 필요할까요?
2. **여러분의 연구 분야**에서 ML이 활용될 수 있는 사례를 생각해보세요.
3. 모델 정확도가 99%라면 **무조건 좋은 것일까요?** (과적합, 데이터 편향 문제)


---
### 🎉 수고하셨습니다!

실습을 통해 데이터를 탐색하고, 시각화하고, 머신러닝 모델을 구축하는 전체 과정을 연습하였습니다. 이 워크플로우는 여러분의 연구 데이터에도 동일하게 적용할 수 있습니다!
